# 🎬 Clipe Musical com IA — Fase 3
## Animação + Lip Sync + Composição Final
### 💚 100% Gratuito · Google Colab GPU T4

| Módulo | Ferramenta | Função |
|---|---|---|
| Animação planos abertos | Ken Burns (OpenCV) | Zoom/pan cinematográfico |
| Animação closes | AnimateDiff | Movimento real do personagem |
| Lip Sync | Wav2Lip | Sincroniza boca com vocal |
| Composição final | FFmpeg | Monta clipe sincronizado |

**O que entra:**
- 📦 ZIP da Fase 1 (`*_fase1.zip`) — storyboard + análise musical
- 📦 ZIP da Fase 2 (`*_fase2_frames.zip`) — frames gerados
- 🎵 Música original `.mp3`

---
⚠️ **Obrigatório:** `Ambiente de execução → Alterar tipo → GPU T4`

## 📦 Célula 1 — Instalação

In [ ]:
print('Instalando dependências...\n')

# AnimateDiff via diffusers
!pip install -q --upgrade diffusers transformers accelerate safetensors

# Wav2Lip — clona o repositório
!git clone -q https://github.com/Rudrabha/Wav2Lip.git /content/Wav2Lip

# Instala requirements do Wav2Lip IGNORANDO a versão do opencv
# (opencv-python==4.1.0.25 não existe mais no pip)
!grep -v 'opencv' /content/Wav2Lip/requirements.txt > /tmp/wav2lip_req.txt
!pip install -q -r /tmp/wav2lip_req.txt
!pip install -q opencv-python-headless  # versão atual compatível

# Pesos do Wav2Lip GAN (melhor qualidade)
import os
os.makedirs('/content/Wav2Lip/checkpoints', exist_ok=True)
os.makedirs('/content/Wav2Lip/temp', exist_ok=True)  # pasta temp obrigatória

!wget -q --show-progress -O /content/Wav2Lip/checkpoints/wav2lip_gan.pth \
    'https://huggingface.co/numz/wav2lip_studio/resolve/main/Wav2lip/wav2lip_gan.pth'

# Detector facial S3FD
os.makedirs('/content/Wav2Lip/face_detection/detection/sfd', exist_ok=True)
!wget -q --show-progress \
    -O /content/Wav2Lip/face_detection/detection/sfd/s3fd.pth \
    'https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth'

# Processamento de vídeo e áudio
!pip install -q librosa soundfile batch-face
!apt-get install -q ffmpeg

print('\n✅ Instalação concluída!')
print('⚠️  Vá em Ambiente de execução → Reiniciar sessão')
print('   Depois rode a partir da Célula 2.')


## 📥 Célula 2 — Upload dos ZIPs e da música

In [ ]:
from google.colab import files
import os, shutil, zipfile, json

BASE_DIR    = '/content/fase3'
INPUT_DIR   = f'{BASE_DIR}/input'
FRAMES_DIR  = f'{BASE_DIR}/frames'
ANIM_DIR    = f'{BASE_DIR}/animados'
SYNC_DIR    = f'{BASE_DIR}/lipsync'
OUTPUT_DIR  = f'{BASE_DIR}/output'
MUSIC_DIR   = f'{BASE_DIR}/musica'

for d in [INPUT_DIR, FRAMES_DIR, ANIM_DIR, SYNC_DIR, OUTPUT_DIR, MUSIC_DIR]:
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d)

# ── ZIP Fase 1 ────────────────────────────────────────────────
print('═'*55)
print('  PASSO 1 — ZIP da Fase 1 (storyboard + análise musical)')
print('═'*55)
up1 = files.upload()
for fn, content in up1.items():
    dest = os.path.join(INPUT_DIR, fn)
    with open(dest, 'wb') as f: f.write(content)
    with zipfile.ZipFile(dest, 'r') as z: z.extractall(INPUT_DIR)
    print(f'✅ Fase 1 extraída: {fn}')

# ── ZIP Fase 2 ────────────────────────────────────────────────
print()
print('═'*55)
print('  PASSO 2 — ZIP da Fase 2 (frames gerados)')
print('═'*55)
up2 = files.upload()
for fn, content in up2.items():
    dest = os.path.join(INPUT_DIR, fn)
    with open(dest, 'wb') as f: f.write(content)
    with zipfile.ZipFile(dest, 'r') as z: z.extractall(FRAMES_DIR)
    print(f'✅ Fase 2 extraída: {fn}')

# ── Música .mp3 ───────────────────────────────────────────────
print()
print('═'*55)
print('  PASSO 3 — Música .mp3')
print('═'*55)
up3 = files.upload()
MUSIC_PATH = None
for fn, content in up3.items():
    dest = os.path.join(MUSIC_DIR, fn)
    with open(dest, 'wb') as f: f.write(content)
    MUSIC_PATH = dest
    print(f'✅ Música: {fn}  ({len(content)/1024/1024:.1f} MB)')

# ── Carrega JSONs ─────────────────────────────────────────────
storyboard = analise_musical = nome_musica = None
for fn in os.listdir(INPUT_DIR):
    path = os.path.join(INPUT_DIR, fn)
    if not fn.endswith('.json'): continue
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    if fn.endswith('_storyboard.json'):
        storyboard  = data
        nome_musica = fn.replace('_storyboard.json', '')
    elif fn.endswith('_analise.json'):
        analise_musical = data

if not storyboard:
    raise ValueError('❌ storyboard JSON não encontrado no ZIP da Fase 1.')

# ── Mapeia frames por cena ────────────────────────────────────
frames_por_cena = {}
for entry in sorted(os.listdir(FRAMES_DIR)):
    cena_path = os.path.join(FRAMES_DIR, entry)
    if os.path.isdir(cena_path) and entry.startswith('cena_'):
        cena_id = int(entry.split('_')[1])
        pngs = sorted([os.path.join(cena_path, f)
                       for f in os.listdir(cena_path) if f.endswith('.png')])
        frames_por_cena[cena_id] = pngs

print(f'\n✅ Projeto: "{nome_musica}"')
print(f'   Cenas no storyboard : {len(storyboard["cenas"])}')
print(f'   BPM                 : {analise_musical["bpm"]}')
print(f'   Duração total       : {analise_musical["duracao_total"]}s')
print(f'   Cenas com frames    : {len(frames_por_cena)}')
for cid, fps in frames_por_cena.items():
    print(f'   Cena {cid:02d}: {len(fps)} frames')


## ⚙️ Célula 3 — Configurações da Fase 3
> Defina aqui antes de rodar qualquer módulo.

In [ ]:
# ── Durações por intensidade (segundos por frame) ──────────────
DURACAO_POR_INTENSIDADE = {
    'alta':  2.5,   # cortes rápidos no refrão
    'média': 4.5,   # ritmo médio
    'baixa': 7.0    # câmera lenta, contemplativo
}

FPS          = 24     # frames por segundo do vídeo final
RESOLUCAO    = (768, 512)  # largura x altura

# ── AnimateDiff — só nos closes ───────────────────────────────
ANIMATEDIFF_STEPS  = 20   # passos (menos = mais rápido)
ANIMATEDIFF_FRAMES = 16   # frames gerados por clipe AnimateDiff

# ── Lip Sync ──────────────────────────────────────────────────
WAV2LIP_RESIZE_FACTOR = 1   # 1=original, 2=metade (mais rápido)
WAV2LIP_NOSMOOTH      = False  # False=suavização ativa (melhor resultado)

# ── Tipos de plano que recebem AnimateDiff vs Ken Burns ───────
PLANOS_ANIMATEDIFF = {'close', 'close-up', 'plano detalhe', 'close up'}
# Qualquer outro plano recebe Ken Burns (zoom/pan)

# ──────────────────────────────────────────────────────────────

bpm      = analise_musical['bpm']
beat_dur = 60.0 / bpm  # duração de uma batida em segundos

print('⚙️  Configurações:')
print(f'   FPS              : {FPS}')
print(f'   Resolução        : {RESOLUCAO[0]}×{RESOLUCAO[1]}px')
print(f'   BPM              : {bpm:.1f}  (batida = {beat_dur:.2f}s)')
print(f'   Duração alta     : {DURACAO_POR_INTENSIDADE["alta"]}s/frame')
print(f'   Duração média    : {DURACAO_POR_INTENSIDADE["média"]}s/frame')
print(f'   Duração baixa    : {DURACAO_POR_INTENSIDADE["baixa"]}s/frame')
print(f'   Planos AnimateDiff: {PLANOS_ANIMATEDIFF}')

# Calcula duração estimada do clipe
dur_estimada = 0
for cena in storyboard['cenas']:
    n_frames = len(frames_por_cena.get(cena['secao_id'], []))
    dur_por_frame = DURACAO_POR_INTENSIDADE[cena['intensidade']]
    dur_estimada += n_frames * dur_por_frame
print(f'\n   Duração estimada do clipe: ~{dur_estimada:.0f}s ({dur_estimada/60:.1f} min)')
print(f'   Duração original da música: {analise_musical["duracao_total"]}s')


## 🎥 Célula 4 — Módulo Ken Burns (planos abertos/médios)

Cada frame vira um clipe com movimento de câmera suave: zoom in, zoom out, pan lateral ou diagonal — definido pela intensidade da cena.

In [ ]:
import cv2
import numpy as np
import os
from PIL import Image

def ken_burns(img_path, out_path, duracao_s, fps, resolucao, intensidade, seed=0):
    """
    Gera um clipe Ken Burns a partir de um frame estático.
    - Alta intensidade: zoom in rápido + leve shake
    - Média: pan lateral suave
    - Baixa: zoom out muito lento (câmera recuando)
    """
    np.random.seed(seed)
    W, H  = resolucao
    n_frames = int(duracao_s * fps)

    img = np.array(Image.open(img_path).convert('RGB').resize((W, H), Image.LANCZOS))
    h, w = img.shape[:2]

    # Parâmetros por intensidade
    if intensidade == 'alta':
        zoom_start, zoom_end = 1.0, 1.12
        pan_x = np.random.choice([-0.03, 0, 0.03])
        pan_y = 0
        shake_mag = 1.5
    elif intensidade == 'média':
        zoom_start, zoom_end = 1.04, 1.0
        pan_x = np.random.choice([-0.05, 0.05])
        pan_y = 0
        shake_mag = 0
    else:  # baixa
        zoom_start, zoom_end = 1.08, 1.0
        pan_x = 0
        pan_y = np.random.choice([-0.02, 0.02])
        shake_mag = 0

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(out_path, fourcc, fps, (W, H))

    for i in range(n_frames):
        t = i / max(n_frames - 1, 1)
        zoom  = zoom_start + (zoom_end - zoom_start) * t
        off_x = int(pan_x * w * t)
        off_y = int(pan_y * h * t)

        # Shake leve nas cenas de alta intensidade
        if shake_mag > 0:
            off_x += int(np.random.uniform(-shake_mag, shake_mag))
            off_y += int(np.random.uniform(-shake_mag, shake_mag))

        # Calcula crop com zoom
        crop_w = int(w / zoom)
        crop_h = int(h / zoom)
        cx = np.clip(w // 2 + off_x, crop_w // 2, w - crop_w // 2)
        cy = np.clip(h // 2 + off_y, crop_h // 2, h - crop_h // 2)

        x1 = cx - crop_w // 2
        y1 = cy - crop_h // 2
        x2 = x1 + crop_w
        y2 = y1 + crop_h

        # Garante bordas
        x1, x2 = max(0, x1), min(w, x2)
        y1, y2 = max(0, y1), min(h, y2)

        cropped = img[y1:y2, x1:x2]
        frame   = cv2.resize(cropped, (W, H), interpolation=cv2.INTER_LINEAR)
        writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

    writer.release()


# ── Processa todos os planos não-close ─────────────────────────
print('🎥 Gerando clipes Ken Burns...\n')

ken_burns_gerados = {}  # cena_id -> lista de paths de clipes

for cena in storyboard['cenas']:
    cena_id    = cena['secao_id']
    intensidade = cena['intensidade']
    tipo_plano  = cena['tipo_de_plano'].lower().strip()
    frames      = frames_por_cena.get(cena_id, [])

    # Pula closes — serão processados pelo AnimateDiff
    if any(p in tipo_plano for p in PLANOS_ANIMATEDIFF):
        print(f'   Cena {cena_id:02d} [{tipo_plano}] → AnimateDiff (próxima célula)')
        continue

    dur_frame = DURACAO_POR_INTENSIDADE[intensidade]
    cena_clips = []

    print(f'   Cena {cena_id:02d} [{tipo_plano}] {intensidade} — {len(frames)} frames × {dur_frame}s')

    for fi, frame_path in enumerate(frames):
        out = os.path.join(ANIM_DIR, f'cena{cena_id:02d}_frame{fi+1:02d}_kb.mp4')
        ken_burns(
            frame_path, out,
            duracao_s=dur_frame, fps=FPS,
            resolucao=RESOLUCAO,
            intensidade=intensidade,
            seed=cena_id * 100 + fi
        )
        cena_clips.append(out)

    ken_burns_gerados[cena_id] = cena_clips
    print(f'      → {len(cena_clips)} clipes gerados')

print(f'\n✅ Ken Burns concluído: {sum(len(v) for v in ken_burns_gerados.values())} clipes')


## 🤖 Célula 5 — Módulo AnimateDiff (closes)

Cada frame de close recebe animação real do personagem via AnimateDiff.

⏳ *Download do modelo ~1.7 GB na primeira vez.*

In [ ]:
import torch
import os
import numpy as np
import cv2
from PIL import Image
from diffusers import MotionAdapter, AnimateDiffPipeline, DDIMScheduler
from diffusers.utils import export_to_video
from transformers import CLIPTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  {device.upper()}')

# ── Carrega AnimateDiff ────────────────────────────────────────
print('\n⬇️  Carregando AnimateDiff...')
adapter = MotionAdapter.from_pretrained(
    'guoyww/animatediff-motion-adapter-v1-5-2',
    torch_dtype=torch.float16
)
anim_pipe = AnimateDiffPipeline.from_pretrained(
    'SG161222/Realistic_Vision_V6.0_B1_noVAE',
    motion_adapter=adapter,
    torch_dtype=torch.float16
).to(device)
anim_pipe.scheduler = DDIMScheduler.from_config(
    anim_pipe.scheduler.config,
    beta_schedule='linear',
    clip_sample=False,
    steps_offset=1
)
anim_pipe.enable_attention_slicing()
print('✅ AnimateDiff pronto!')

# Tokenizer para truncagem
tokenizer_clip = CLIPTokenizer.from_pretrained('openai/clip-vit-base-patch32')
def truncar_prompt(texto, max_tokens=75):
    ids = tokenizer_clip.encode(texto, add_special_tokens=False)
    if len(ids) <= max_tokens: return texto
    return tokenizer_clip.decode(ids[:max_tokens], skip_special_tokens=True)

NEGATIVE_ANIM = (
    'blurry, low quality, distorted, ugly, bad anatomy, extra limbs, '
    'watermark, text, poorly drawn face, mutation, deformed'
)

# ── Processa closes com AnimateDiff ───────────────────────────
print('\n🤖 Gerando animações nos closes...\n')

animatediff_gerados = {}

for cena in storyboard['cenas']:
    cena_id    = cena['secao_id']
    intensidade = cena['intensidade']
    tipo_plano  = cena['tipo_de_plano'].lower().strip()
    frames      = frames_por_cena.get(cena_id, [])

    # Só processa closes
    if not any(p in tipo_plano for p in PLANOS_ANIMATEDIFF):
        continue

    dur_frame = DURACAO_POR_INTENSIDADE[intensidade]
    n_frames_video = max(8, min(ANIMATEDIFF_FRAMES, int(dur_frame * FPS)))
    prompt = truncar_prompt(cena['prompt_sd'])

    print(f'   Cena {cena_id:02d} [{tipo_plano}] {intensidade} — {len(frames)} frames')
    cena_clips = []

    for fi, frame_path in enumerate(frames):
        seed      = 42 + cena_id * 100 + fi
        generator = torch.Generator(device=device).manual_seed(seed)

        output = anim_pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_ANIM,
            num_frames=n_frames_video,
            num_inference_steps=ANIMATEDIFF_STEPS,
            guidance_scale=7.0,
            width=RESOLUCAO[0],
            height=RESOLUCAO[1],
            generator=generator
        )

        # Salva como mp4
        tmp_frames = output.frames[0]  # lista de PIL
        out_path   = os.path.join(ANIM_DIR, f'cena{cena_id:02d}_frame{fi+1:02d}_ad.mp4')

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(out_path, fourcc, FPS, RESOLUCAO)
        for pil_f in tmp_frames:
            bgr = cv2.cvtColor(np.array(pil_f.resize(RESOLUCAO)), cv2.COLOR_RGB2BGR)
            writer.write(bgr)
        writer.release()

        cena_clips.append(out_path)
        print(f'      Frame {fi+1}/{len(frames)} animado')

    animatediff_gerados[cena_id] = cena_clips
    print(f'      → {len(cena_clips)} clipes AnimateDiff gerados')

print(f'\n✅ AnimateDiff concluído: {sum(len(v) for v in animatediff_gerados.values())} clipes')


## 🔗 Célula 6 — Montagem do clipe sem áudio (pré lip sync)

Junta todos os clipes animados na ordem correta do storyboard com FFmpeg.
O resultado é o clipe mudo que será passado para o Wav2Lip.

In [ ]:
import subprocess, os

# ── Monta lista de clipes na ordem do storyboard ──────────────
lista_clipes = []

for cena in storyboard['cenas']:
    cena_id    = cena['secao_id']
    tipo_plano = cena['tipo_de_plano'].lower().strip()
    eh_close   = any(p in tipo_plano for p in PLANOS_ANIMATEDIFF)

    if eh_close:
        clips = animatediff_gerados.get(cena_id, [])
        tipo  = 'AnimateDiff'
    else:
        clips = ken_burns_gerados.get(cena_id, [])
        tipo  = 'KenBurns'

    for clip in clips:
        lista_clipes.append(clip)

    print(f'   Cena {cena_id:02d} [{tipo_plano}] → {tipo}: {len(clips)} clipes')

if not lista_clipes:
    raise ValueError('❌ Nenhum clipe gerado. Verifique as células 4 e 5.')

# ── Cria arquivo de lista para o FFmpeg ───────────────────────
lista_path = os.path.join(OUTPUT_DIR, 'lista_clipes.txt')
with open(lista_path, 'w') as f:
    for clip in lista_clipes:
        f.write(f"file '{clip}'\n")

# ── Concatena com FFmpeg ───────────────────────────────────────
pre_lipsync_path = os.path.join(OUTPUT_DIR, f'{nome_musica}_pre_lipsync.mp4')

print(f'\n🔗 Concatenando {len(lista_clipes)} clipes com FFmpeg...')
cmd = [
    'ffmpeg', '-y',
    '-f', 'concat',
    '-safe', '0',
    '-i', lista_path,
    '-c:v', 'libx264',
    '-preset', 'fast',
    '-crf', '18',
    '-pix_fmt', 'yuv420p',
    '-vf', f'scale={RESOLUCAO[0]}:{RESOLUCAO[1]}',
    pre_lipsync_path
]
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode != 0:
    print('❌ Erro FFmpeg:')
    print(result.stderr[-2000:])
else:
    size = os.path.getsize(pre_lipsync_path) / (1024*1024)
    print(f'✅ Clipe pré-lipsync: {pre_lipsync_path}  ({size:.1f} MB)')

    # Checa duração
    probe = subprocess.run(
        ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
         '-of', 'default=noprint_wrappers=1:nokey=1', pre_lipsync_path],
        capture_output=True, text=True
    )
    dur = float(probe.stdout.strip())
    print(f'   Duração do clipe : {dur:.1f}s')
    print(f'   Duração da música: {analise_musical["duracao_total"]}s')


## 👄 Célula 7 — Lip Sync com Wav2Lip

Sincroniza os movimentos de boca do personagem com o **vocal isolado** pelo Demucs na Fase 1.

⏳ *Processa frame a frame — pode levar 10-20 min.*

In [ ]:
import subprocess, os, shutil, math

os.makedirs('/content/Wav2Lip/temp', exist_ok=True)
CHUNKS_DIR = '/content/Wav2Lip/temp/chunks'
os.makedirs(CHUNKS_DIR, exist_ok=True)

# ── Patch librosa ─────────────────────────────────────────────
audio_py = '/content/Wav2Lip/audio.py'
with open(audio_py, 'r') as f:
    codigo = f.read()
antigo = 'return librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=hp.num_mels,'
novo   = 'return librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft, n_mels=hp.num_mels,'
if antigo in codigo:
    with open(audio_py, 'w') as f:
        f.write(codigo.replace(antigo, novo))
    print('✅ Patch librosa aplicado!')
else:
    print('✅ Patch já aplicado.')

# ── Vocal / áudio ─────────────────────────────────────────────
VOCAL_PATH = analise_musical.get('vocal_path')
if not VOCAL_PATH or not os.path.exists(str(VOCAL_PATH)):
    print('⚠️  Usando música completa como áudio.')
    VOCAL_PATH = MUSIC_PATH
else:
    print(f'✅ Vocal: {VOCAL_PATH}')

# Sanitiza caminhos
audio_safe = '/content/Wav2Lip/temp/input_audio.mp3'
shutil.copy2(VOCAL_PATH, audio_safe)

# ── Descobre duração do vídeo ─────────────────────────────────
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
     '-of', 'default=noprint_wrappers=1:nokey=1', pre_lipsync_path],
    capture_output=True, text=True
)
duracao_video = float(probe.stdout.strip())

# ── Processa em chunks de 20s para economizar RAM ─────────────
CHUNK_DUR = 20  # segundos por chunk — ajuste se ainda estourar RAM
n_chunks  = math.ceil(duracao_video / CHUNK_DUR)

print(f'\n👄 Wav2Lip em {n_chunks} chunks de {CHUNK_DUR}s')
print(f'   Duração total: {duracao_video:.1f}s\n')

chunks_ok = []

for i in range(n_chunks):
    t_start = i * CHUNK_DUR
    t_end   = min((i + 1) * CHUNK_DUR, duracao_video)
    dur     = t_end - t_start

    chunk_video = os.path.join(CHUNKS_DIR, f'chunk_{i:03d}_video.mp4')
    chunk_audio = os.path.join(CHUNKS_DIR, f'chunk_{i:03d}_audio.mp3')
    chunk_out   = os.path.join(CHUNKS_DIR, f'chunk_{i:03d}_sync.mp4')

    # Recorta chunk de vídeo
    subprocess.run([
        'ffmpeg', '-y', '-ss', str(t_start), '-t', str(dur),
        '-i', pre_lipsync_path,
        '-c:v', 'libx264', '-preset', 'fast', '-crf', '18',
        chunk_video
    ], capture_output=True)

    # Recorta chunk de áudio
    subprocess.run([
        'ffmpeg', '-y', '-ss', str(t_start), '-t', str(dur),
        '-i', audio_safe, chunk_audio
    ], capture_output=True)

    print(f'   Chunk {i+1}/{n_chunks}  [{t_start:.0f}s → {t_end:.0f}s]  processando...')

    result = subprocess.run(
        [
            'python', 'inference.py',
            '--checkpoint_path', 'checkpoints/wav2lip_gan.pth',
            '--face',    chunk_video,
            '--audio',   chunk_audio,
            '--outfile', chunk_out,
            '--resize_factor', str(WAV2LIP_RESIZE_FACTOR),
            '--pads', '0', '10', '0', '0',
            '--fps', str(FPS),
            '--nosmooth',  # mais rápido e menos RAM
        ],
        capture_output=True, text=True,
        cwd='/content/Wav2Lip'
    )

    if result.returncode != 0:
        print(f'   ❌ Chunk {i+1} falhou:')
        print(result.stderr[-1000:])
    else:
        chunks_ok.append(chunk_out)
        print(f'   ✅ Chunk {i+1}/{n_chunks} concluído')

    # Limpa arquivos temporários do chunk para liberar RAM
    for tmp in [chunk_video, chunk_audio]:
        if os.path.exists(tmp): os.remove(tmp)

# ── Concatena todos os chunks sincronizados ───────────────────
lipsync_path = os.path.join(OUTPUT_DIR, f'{nome_musica}_lipsync.mp4')

if not chunks_ok:
    raise RuntimeError('❌ Nenhum chunk gerado com sucesso.')

print(f'\n🔗 Concatenando {len(chunks_ok)} chunks...')
lista_chunks = os.path.join(CHUNKS_DIR, 'lista.txt')
with open(lista_chunks, 'w') as f:
    for c in chunks_ok:
        f.write(f"file '{c}'\n")

subprocess.run([
    'ffmpeg', '-y',
    '-f', 'concat', '-safe', '0',
    '-i', lista_chunks,
    '-c:v', 'libx264', '-preset', 'fast', '-crf', '18',
    '-pix_fmt', 'yuv420p',
    lipsync_path
], capture_output=True)

size = os.path.getsize(lipsync_path) / (1024*1024)
print(f'\n✅ Lip sync concluído!')
print(f'   Chunks processados : {len(chunks_ok)}/{n_chunks}')
print(f'   Saída              : {lipsync_path}  ({size:.1f} MB)')


## 🎵 Célula 8 — Composição final com áudio original

Mistura o clipe com lip sync + áudio original da música.
O vídeo é cortado ou estendido para casar exatamente com a duração da música.

In [ ]:
import subprocess, os

final_path = os.path.join(OUTPUT_DIR, f'{nome_musica}_CLIPE_FINAL.mp4')
dur_musica = analise_musical['duracao_total']

print(f'🎵 Compondo clipe final...')
print(f'   Vídeo (lip sync) : {lipsync_path}')
print(f'   Áudio original   : {MUSIC_PATH}')
print(f'   Duração alvo     : {dur_musica}s')

# Combina vídeo (sem áudio) com música original
# -shortest: para no menor dos dois (música ou vídeo)
# -t: limita à duração exata da música
cmd = [
    'ffmpeg', '-y',
    '-i', lipsync_path,          # vídeo com lip sync
    '-i', MUSIC_PATH,            # áudio original
    '-map', '0:v',               # vídeo do primeiro input
    '-map', '1:a',               # áudio do segundo input
    '-t', str(dur_musica),       # corta na duração da música
    '-c:v', 'libx264',
    '-preset', 'slow',           # melhor compressão para entrega
    '-crf', '16',                # alta qualidade
    '-c:a', 'aac',
    '-b:a', '192k',
    '-pix_fmt', 'yuv420p',
    '-movflags', '+faststart',   # otimiza para streaming
    final_path
]

result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode != 0:
    print('❌ Erro FFmpeg:')
    print(result.stderr[-2000:])
else:
    size = os.path.getsize(final_path) / (1024*1024)
    print(f'\n✅ CLIPE FINAL gerado!')
    print(f'   {final_path}')
    print(f'   Tamanho: {size:.1f} MB')

    probe = subprocess.run(
        ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
         '-of', 'default=noprint_wrappers=1:nokey=1', final_path],
        capture_output=True, text=True
    )
    print(f'   Duração: {float(probe.stdout.strip()):.1f}s')


## 💾 Célula 9 — Download do clipe final

In [ ]:
from google.colab import files
import os

print(f'📥 Baixando: {os.path.basename(final_path)}')
files.download(final_path)

print('\n🎉 Pipeline completo!')
print('═'*50)
print(f'  Projeto    : {nome_musica}')
print(f'  Fase 1     : Análise musical + Storyboard (Groq)')
print(f'  Fase 2     : Frames gerados (SD + IP-Adapter)')
print(f'  Fase 3     : Animação + Lip Sync + FFmpeg')
print('═'*50)


---
## 📋 Resumo do pipeline

```
ZIP Fase 1 + ZIP Fase 2 + .mp3
         ↓
  Planos abertos → Ken Burns (zoom/pan/shake)
  Closes         → AnimateDiff (movimento real)
         ↓
  FFmpeg concat → clipe mudo
         ↓
  Wav2Lip → lip sync com vocal
         ↓
  FFmpeg final → + áudio original → CLIPE_FINAL.mp4
```

## 🔧 Ajustes rápidos

| Quer mudar | Edite na Célula |
|---|---|
| Velocidade de corte | `DURACAO_POR_INTENSIDADE` — Célula 3 |
| Quais planos são closes | `PLANOS_ANIMATEDIFF` — Célula 3 |
| Intensidade da animação | `ANIMATEDIFF_STEPS` / `FRAMES` — Célula 3 |
| Qualidade lip sync | `WAV2LIP_NOSMOOTH` — Célula 3 |